# Notebook 04 — Evaluation Metrics

Compare Cellpose and Baysor against the Xenium DAPI-derived nucleus boundaries (ground truth reference) and against each other.

**Metrics computed:**
- IoU and Dice score vs. DAPI reference polygons
- Matched cell rate (recall of reference nuclei)
- Transcript assignment stats (cells detected, unassigned fraction)
- Cell size distributions
- Parameter sweep stability (CV of cell count)

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile
import json
from pathlib import Path
from shapely.geometry import Polygon
from skimage import measure
from tqdm import tqdm

from src.evaluation.metrics import (
    match_cells_to_reference,
    transcript_assignment_stats,
    build_summary_table,
)
from src.visualization.spatial_plots import (
    plot_transcripts_per_cell,
    plot_iou_comparison,
    compare_segmentations,
)

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")

## 1. Load Reference (DAPI) Nucleus Polygons

In [ ]:
nucleus_boundaries = pd.read_csv(DATA_DIR / "nucleus_boundaries.csv.gz", compression="gzip")

def boundaries_to_polygons(df: pd.DataFrame, id_col="cell_id", x_col="vertex_x", y_col="vertex_y") -> list:
    polys = []
    for cell_id, group in df.groupby(id_col):
        coords = list(zip(group[x_col], group[y_col]))
        if len(coords) >= 3:
            poly = Polygon(coords)
            if poly.is_valid and poly.area > 0:
                polys.append(poly)
    return polys

ref_polys = boundaries_to_polygons(nucleus_boundaries)
print(f"Reference nuclei: {len(ref_polys):,}")

## 2. Load Cellpose Polygons

In [ ]:
masks = np.load(DATA_DIR / "cellpose_masks.npy")

PIXEL_SIZE = 0.2125  # µm per pixel

def masks_to_polygons_um(mask_array: np.ndarray, pixel_size: float) -> list:
    polys = []
    for region in measure.regionprops(mask_array):
        contours = measure.find_contours(mask_array == region.label, 0.5)
        if not contours:
            continue
        contour = max(contours, key=len)
        # Convert row/col (pixels) to µm to match reference coordinates
        xy_um = contour[:, [1, 0]] * pixel_size  # (x_um, y_um)
        poly = Polygon(xy_um)
        if poly.is_valid and poly.area > 0:
            polys.append(poly)
    return polys

cellpose_polys = masks_to_polygons_um(masks, PIXEL_SIZE)
print(f"Cellpose polygons: {len(cellpose_polys):,}")

## 3. Load Baysor Polygons

In [ ]:
with open(DATA_DIR / "baysor_output" / "segmentation_polygons.json") as f:
    geojson = json.load(f)

baysor_polys = []
for feature in geojson["features"]:
    geom = feature["geometry"]
    if geom["type"] == "Polygon":
        coords = geom["coordinates"][0]
        poly = Polygon(coords)
        if poly.is_valid and poly.area > 0:
            baysor_polys.append(poly)

print(f"Baysor polygons: {len(baysor_polys):,}")

## 4. IoU / Dice vs. DAPI Reference

We use a random spatial subsample (FOV crop) to keep matching tractable.

In [ ]:
# Crop to a 500×500 µm region of interest for tractable polygon matching
X_MIN, X_MAX = 1000, 1500
Y_MIN, Y_MAX = 1000, 1500

def crop_polys(polys, x_min, x_max, y_min, y_max):
    from shapely.geometry import box
    roi = box(x_min, y_min, x_max, y_max)
    return [p for p in polys if p.centroid.x >= x_min and p.centroid.x <= x_max
            and p.centroid.y >= y_min and p.centroid.y <= y_max]

ref_crop = crop_polys(ref_polys, X_MIN, X_MAX, Y_MIN, Y_MAX)
cp_crop = crop_polys(cellpose_polys, X_MIN, X_MAX, Y_MIN, Y_MAX)
by_crop = crop_polys(baysor_polys, X_MIN, X_MAX, Y_MIN, Y_MAX)

print(f"ROI — Reference: {len(ref_crop)}, Cellpose: {len(cp_crop)}, Baysor: {len(by_crop)}")

In [ ]:
print("Matching Cellpose → reference...")
cp_match = match_cells_to_reference(cp_crop, ref_crop, iou_threshold=0.3)

print("Matching Baysor → reference...")
by_match = match_cells_to_reference(by_crop, ref_crop, iou_threshold=0.3)

print("\n--- Cellpose vs. DAPI Reference ---")
for k, v in cp_match.items():
    if not isinstance(v, list):
        print(f"  {k}: {v:.4f}")

print("\n--- Baysor vs. DAPI Reference ---")
for k, v in by_match.items():
    if not isinstance(v, list):
        print(f"  {k}: {v:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

plot_iou_comparison(
    {"Cellpose": cp_match["matched_iou"], "Baysor": by_match["matched_iou"]},
    ax=axes[0],
)

# Dice
methods = ["Cellpose", "Baysor"]
dice_means = [np.mean(cp_match["matched_dice"]), np.mean(by_match["matched_dice"])]
dice_stds = [np.std(cp_match["matched_dice"]), np.std(by_match["matched_dice"])]
bars = axes[1].bar(methods, dice_means, yerr=dice_stds, color=["steelblue", "seagreen"], capsize=5, alpha=0.85)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("Mean Dice Score")
axes[1].set_title("Segmentation Accuracy (Dice)")
for bar, mean in zip(bars, dice_means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, mean + 0.02, f"{mean:.2f}", ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "iou_dice_comparison.png", dpi=150)
plt.show()

## 5. Transcript Assignment Comparison

In [ ]:
xenium_transcripts = pd.read_parquet(DATA_DIR / "transcripts_filtered.parquet")
cellpose_transcripts = pd.read_parquet(DATA_DIR / "transcripts_cellpose.parquet")
baysor_transcripts = pd.read_parquet(DATA_DIR / "transcripts_baysor.parquet")

xenium_stats = transcript_assignment_stats(xenium_transcripts, cell_col="cell_id")
cellpose_stats = transcript_assignment_stats(cellpose_transcripts, cell_col="cellpose_cell_id")
baysor_stats = transcript_assignment_stats(baysor_transcripts, cell_col="cell")

summary = build_summary_table({
    "Xenium (10x)": xenium_stats,
    "Cellpose": cellpose_stats,
    "Baysor": baysor_stats,
})

print(summary.to_string())
summary.to_csv(RESULTS_DIR / "method_summary_stats.csv")

## 6. Transcripts per Cell Distribution

In [ ]:
xenium_tpc = xenium_transcripts[xenium_transcripts["cell_id"].notna()].groupby("cell_id").size()
cellpose_tpc = cellpose_transcripts[cellpose_transcripts["cellpose_cell_id"] > 0].groupby("cellpose_cell_id").size()
baysor_tpc = baysor_transcripts[baysor_transcripts["cell"] != 0].groupby("cell").size()

fig, ax = plt.subplots(figsize=(8, 5))
plot_transcripts_per_cell(
    {"Xenium (10x)": xenium_tpc, "Cellpose": cellpose_tpc, "Baysor": baysor_tpc},
    ax=ax,
)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "transcripts_per_cell_distribution.png", dpi=150)
plt.show()

## 7. Cell Size Distribution

In [ ]:
cp_areas = [p.area for p in cellpose_polys]
by_areas = [p.area for p in baysor_polys]
ref_areas = [p.area for p in ref_polys]

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(ref_areas, bins=60, alpha=0.5, label="Xenium DAPI (ref)", color="gold", density=True)
ax.hist(cp_areas, bins=60, alpha=0.6, label="Cellpose", color="steelblue", density=True)
ax.hist(by_areas, bins=60, alpha=0.6, label="Baysor", color="seagreen", density=True)
ax.set_xlabel("Cell area (µm²)")
ax.set_ylabel("Density")
ax.set_title("Cell Size Distribution Across Methods")
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "cell_size_distribution.png", dpi=150)
plt.show()

## 8. Side-by-Side Segmentation Map (FOV Crop)

In [ ]:
dapi = tifffile.imread(DATA_DIR / "morphology.ome.tif")
dapi_norm = (dapi - dapi.min()) / (dapi.max() - dapi.min())

# Crop DAPI image to same ROI (pixels)
r0 = int(Y_MIN / PIXEL_SIZE)
r1 = int(Y_MAX / PIXEL_SIZE)
c0 = int(X_MIN / PIXEL_SIZE)
c1 = int(X_MAX / PIXEL_SIZE)
dapi_crop = dapi_norm[r0:r1, c0:c1]

fig = compare_segmentations(
    {"Xenium (DAPI ref)": ref_crop, "Cellpose": cp_crop, "Baysor": by_crop},
    image=dapi_crop,
)
fig.savefig(RESULTS_DIR / "side_by_side_segmentation.png", dpi=150)
plt.show()